Transformers 라이브러리

인코더 기반 모델: 예시 모델 Multiple Choice
- 문제 예시) 여러개 입력이 주어졌을때 주어진 문장중 옳은 문장을 고르는 객관식 문제, 
- 입력 프롬프트 한개와 그에 대응하는 n개 후보로 구성됨
- 프롬프트 + 답변1, 프롬프트 + 답변2,...프롬프트 + 답변n 목록을 모두 입력 받아 후보중 무엇이 정답인지 출력

1. klue/bert-base 기본 모델: BertForMultipleChoice
- 추론 : 수능 국어 문제 풀이 - 5지선다형(보기 5개)

2. klue/bert-base 기본 모델: BertForMultipleChoice
- 파인튜닝 : 수능 국어 문제 풀이 - 5지선다형(보기 5개)
- 학습용 데이터셋 준비: 데이터셋(수능 문제 풀이 HAERAE-HUB/csatqa)
- 데이터셋 전처리(토크나이징), 샘플링
- 다중선택(MultipleChoice) 모데을 위한 동적 패딩 전용 커스텀 데이터 콜레이터(Data Collator)
- 트레이너(Trainer) 설정 
- 학습 실행
- 모델 저장
- 모델 로드 및 모델 추론
- 허깅페이스 모델 업로드

In [1]:
# GPU 테스트
import torch
print("GPU 사용 가능 여부:", torch.cuda.is_available())
print("사용 중인 GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

GPU 사용 가능 여부: True
사용 중인 GPU: NVIDIA GeForce RTX 3060 Laptop GPU


In [2]:
# 인코더 기반 모델
# 1. klue/bert-base 기본 모델: BertForMultipleChoice
# 예를 들어, "지문"과 "보기 1, 2, 3, 4, 5"가 주어졌을 때 각각의 보기가 정답일 확률(점수)을 계산하여 
# 가장 높은 점수를 가진 보기를 선택할 수 있게 만들어 준다
import torch
from transformers import AutoTokenizer, AutoModelForMultipleChoice

# 모델 이름 지정 및 사전학습 모델 로드
model_name = "klue/bert-base"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForMultipleChoice.from_pretrained(model_name)

# 모델 출력 확인
print(f"model : {model}")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForMultipleChoice LOAD REPORT from: klue/bert-base
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


model : BertForMultipleChoice(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(32000, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True, bias=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,),

In [3]:
# 인코더 기반 모델
# 1. klue/bert-base 기본 모델: BertForMultipleChoice
# 추론 : 수능 국어 문제 풀이 - 5지선다형(보기 5개)
import torch
from transformers import AutoTokenizer, AutoModelForMultipleChoice

# 1. 디바이스 설정
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

# 2. 모델 이름 지정 및 사전학습 모델 로드
model_name = "klue/bert-base"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForMultipleChoice.from_pretrained(model_name)
print(f"model: {model}")

# 3. 모델 -> 디바이스 이동
model.to(device)

# 4. 평가/추론시 설정: 드롭아웃 비활성화, 정규화 레이어 고정
model.eval()

# 5. 수능 지문 및 5지선다 보기 설정
passage = "다음 글을 읽고 물음에 답하시오. (중략) 화자는 흐르는 강물을 바라보며 세월의 무상함과 인생의 덧없음을 달래고 있다."
choices = [
    "① 현실의 타협과 물질적 성취를 추구하고 있다.",
    "② 자연 친화적 삶을 통해 내면의 평온을 찾고 있다.",
    "③ 세월의 흐름에 따른 삶의 덧없음을 달래고 있다.",
    "④ 문명의 이기를 비판하며 도시로 회귀하고 있다.",
    "⑤ 과거의 역사적 사건을 객관적으로 고증하고 있다."
]

# 6. 토크나이징 (질문 + 답변 결합)
# - choices 리스트에는 1~5번까지의 보기(답변)가 들어 있음
# - 지문과 결합하는 작업을 5번 반복, 예) 1번째 반복 지문 + 1번 보기, 2번째 반복 지문 + 2번 보기 ...
encoded_inputs = [
    # text=passage, text_pair=choice 질문과 답변의 결합
    # 허깅페이스 토크나이저에서 text/text_pair 인자를 받으면 두문장을 하나의 문장처럼 이어 붙이면서 중간과 끝에 특수 토큰[CLS] [SEP] 삽입해준다
    # 예) [CLS] 지문 내용... [SEP] 3번 보기 내용... [SEP]
    tokenizer(
        text=passage,
        text_pair=choice,
        return_tensors="pt",
        truncation=True,
        padding="max_length",
        max_length=128,
    # 리스트 컴프리헨션(List Comprehension)이 실행되고 나면, encoded_inputs라는 리스트 안에
    # 총 5개의 토큰화된 결과 묶음이 들어가게 된다.
    ) for choice in choices
]

# 7. 3차원 텐서 구성 및 디바이스 리동
# 5개의 보기 텐서를 하나로 묶고([batch_size,num_choices,sequence] 예)[1,5,128]), 모델과 같은 디바이스로 보낸다
# 3차원 텐서 구성 작업 설명
# - .squeeze(0): [1,128] 불필요한 첫번째 차원(1)을 제거하여 -> [128] 모양의 1차원 텐서로 만들어준다
# - torch.stack(tensors=..., dim=0): 리스트안에 5개의 텐서[128]를 새로운 차원(dim=0)을 기준으로 수직으로 쌓아서 하나의 큰 텐서로 합쳐준다
# - 예)[128] 1차원 텐서 -> [5,128] 2차원 텐서로 만든다, 여기서 5는 보기 개수(num_choices=5)가 된다
# - .unsqueeze(0): 텐서의 맨 앞에 크기가 1인 차원(dim=0)을 새롭게 끼워 넣는다
# - 예)[5,128] 2차원 텐서 -> [1,5,128] 3차원 텐서로 만든다, 여기서 1은 배치 크기(batch_size=1)가 된다, 지금 한번에 처리할 문항(배치)이 1개 있다
# - .to(device): [1,5,128] 최종 3차원 텐서 -> 디비이스 이동
input_ids = torch.stack(
    tensors=[item['input_ids'].squeeze(0) for item in encoded_inputs],
    dim=0,
).unsqueeze(0).to(device)
print(f"input_ids: {input_ids}")
print(f"input_ids shape: {input_ids.shape}")

attention_mask = torch.stack(
    tensors=[item['attention_mask'].squeeze(0) for item in encoded_inputs],
    dim=0,
).unsqueeze(0).to(device)
print(f"attention_mask: {attention_mask}")
print(f"attention_mask shape: {attention_mask.shape}")

# 8. 모델 추론
with torch.no_grad(): # 미분 사용하지 않음
    # 모델 추론 전체 결과
    outputs = model(
        input_ids=input_ids,
        attention_mask=attention_mask,
    )
    print(f"outputs: {outputs}")

    # logits 결과: [1,5] (보기 5개 각각의 점수)
    logits = outputs.logits
    print(f"logits: {logits}, {logits.shape}")

# 9. 결과 확인 및 정답 예측
# - torch.softmax(): logits [1,5] (batch_size, num_choices) 소프트맥스 함수는 어떤 축(Dimension)을 기준으로 확률의 합이 1이 되도록 만들것인가를 결정해야 된다
# - 예)5개의 보기(1~5번) 각각의 정답일 확률을 계산해서 그 5개의 확률 합이 1.0(100%) 되게 한다
# - .squeeze(): logits 모양 [1,5] 2차원 텐서 -> [5] 1차원 텐서로 만든다, 불필요한 차원을 제거
# - .tolist(): 텐서 -> 리스트 자료형으로 최종 만든다
# 소프트맥스 계산
probs = torch.softmax(
    input=logits,
    dim=-1
).squeeze().tolist()
print(f"probs: {probs}")

# 로짓값: 차원축소,리스트 형변환
scores = logits.squeeze().tolist()
print(f"scores: {scores}")

# 가장 높은 로짓값의 인덱스 번호
predicted_idx = torch.argmax(
    input=logits,
    dim=-1,
).item()
print(f"predicted_idx: {predicted_idx}")

print("\n=== 5지선다 추론 결과 ===")
for idx, (choice, score, prob) in enumerate(zip(choices, scores, probs)):
    print(f"{choice} -> 점수(logits): {score:.4f} | 확률: {prob * 100:.4f}%")

print(f"\n모델이 선택한 정답: {predicted_idx + 1}번 ({choices[predicted_idx]})")

Device: cuda


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForMultipleChoice LOAD REPORT from: klue/bert-base
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


model: BertForMultipleChoice(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(32000, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True, bias=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), 

In [4]:
# 2. klue/bert-base 기본 모델: BertForMultipleChoice
# - 파인튜닝 : 수능 국어 문제 풀이 - 5지선다형(보기 5개)
# - 학습용 데이터셋 준비: 데이터셋(수능 문제 풀이 HAERAE-HUB/csatqa)
# - 데이터셋 전처리(토크나이징), 샘플링
# - 트레이너(Trainer) 설정 
# - 학습 실행
# - 모델 저장
# - 모델 로드 및 모델 추론
# - 허깅페이스 모델 업로드
# 1. 학습용 데이터셋 준비: 데이터셋(수능 문제 풀이 HAERAE-HUB/csatqa)
from datasets import load_dataset
import torch
from transformers import AutoTokenizer

# 2. 디바이스 설정
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

# 3. 모델 이름 지정 및 사전학습 모델 로드
model_name = "klue/bert-base"
tokenizer = AutoTokenizer.from_pretrained(model_name)
print(f"tokenizer: {tokenizer}")

Device: cuda
tokenizer: BertTokenizer(name_or_path='klue/bert-base', vocab_size=32000, model_max_length=512, padding_side='right', truncation_side='right', special_tokens={'unk_token': '[UNK]', 'sep_token': '[SEP]', 'pad_token': '[PAD]', 'cls_token': '[CLS]', 'mask_token': '[MASK]'}, added_tokens_decoder={
	0: AddedToken("[PAD]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	1: AddedToken("[UNK]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	2: AddedToken("[CLS]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	3: AddedToken("[SEP]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	4: AddedToken("[MASK]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
})


In [5]:
# 4. HAERAE-HUB/csatqa 데이터셋 로드
from datasets import load_dataset

# full config로 데이터셋 로드, full 은 전체 통합본을 쓰거나, GR 은 국어 영역 관련 선택
config_name = "full" # full 또는 GR

# 최신 허깅페이스 datasets 라이브러리는 보안상의 이유로 데이터셋 내부의 커스텀 파이썬 스크립트 실행을 기본적으로 차단
# 이 때문에 csatqa.py 파일을 불러오지 못해 RuntimeError 발생 -> trust_remote_code=True
dataset = load_dataset("HAERAE-HUB/csatqa", name=config_name, trust_remote_code=True)
print(f"dataset 원본 데이터셋 구조 : {dataset}")

# test 데이터셋을 8:2 비율로 train/eval(검증용)로 분할
# seed를 고정해 두면 코드를 다시 실행해도 똑같은 방식으로 쪼개진다
split_dataset = dataset['test'].train_test_split(test_size=0.2, seed=42)

train_dataset = split_dataset['train'] # 학습용(약 748개)
eval_dataset = split_dataset['test'] # 평가용(약 188개)
print("\n데이터 갯수 확인")
print(f"train_dataset: {len(train_dataset)}")
print(f"eval_dataset: {len(eval_dataset)}")

print("\n첫번째 학습 데이터 구조 샘플")
print(f"train_dataset[0]: {train_dataset[0]}")

dataset 원본 데이터셋 구조 : DatasetDict({
    test: Dataset({
        features: ['question', 'context', 'option#1', 'option#2', 'option#3', 'option#4', 'option#5', 'gold', 'category', 'human_performance'],
        num_rows: 936
    })
})

데이터 갯수 확인
train_dataset: 748
eval_dataset: 188

첫번째 학습 데이터 구조 샘플
train_dataset[0]: {'question': '. [A]와 [B]를 비교한 내용으로 가장 적절한 것은?', 'context': '다음 글을 읽고 물음에 답하시오.\n\n<par>(가) 자화상(自畵像)\n윤동주\n산모퉁이를 돌아 논가 외딴 우물을 홀로 \n찾아가선 가만히 들여다봅니다.\n우물 속에는 달이 밝고 구름이 흐르고\n하늘이 펼치고 파아란 바람이 불고 가을이 있습니다.\n그리고 한 사나이가 있습니다.\n어쩐지 그 사나이가 미워져 돌아갑니다.\n돌아가다 생각하니 그 사나이가 가엾어집니다. 도로 가 \n들여다보니 사나이는 그대로 있습니다.\n다시 그 사나이가 미워져 돌아갑니다.\n돌아가다 생각하니 그 사나이가 그리워집니다.\n우물 속에는 달이 밝고 구름이 흐르고 하늘이 펼치고 파아란\n바람이 불고 가을이 있고 추억처럼 사나이가 있습니다.<par/>\n<par>(나) 선제리 아낙네들\n고은\n<sent>먹밤중 한밤중 새터 중뜸 개들이 시끌짝하게 짖어댄다\n이 개 짖으니 저 개도 짖어\n들 건너 갈메 개까지 덩달아 짖어댄다\n이런 개 짖는 소리 사이로\n언뜻언뜻 까 여 다 여 따위 말끝이 들린다\n밤 기러기 드높게 날며\n추운 땅으로 떨어뜨리는 소리하고 남이 아니다\n앞서거니 뒤서거니 의좋은 그 소리하고 남이 아니다<sent/>\n콩밭 김칫거리\n아쉬울 때 마늘 한 접 이고 가서\n군산 묵은장 가서 팔고 오는 선제리 아낙네들\n팔

In [ ]:
# 5. 데이터셋 전처리(토크나이징), 샘플링
# 3차원 텐서로 변환하여 전처리(토크나이징) 단계
# BertForMultipleChoice 모델은 일반적인 분류 모델과 달리 입력데이터가 [batch_size, num_choice, sequence] 구조를 가져야 한다
from transformers import AutoTokenizer

# 모델 이름 지정 및 사전학습 모델 로드
model_name = "klue/bert-base"
tokenizer = AutoTokenizer.from_pretrained(model_name)

# 전처리(토크나이징) 함수
# 지문과 5개의 보기를 각각 짝지어 토큰화, BertForMultipleChoice 모델이 요구하는 3차원 텐서 형태로 변환
def preprocess_function(examples):
    # castqa 데이터셋의 실제 필드명에 맞추어 키값 지정
    batch_input_ids = []
    batch_attention_mask = []
    batch_labels = []

    for i in range(len(examples['question'])):
        # context 지문, question 질문 추출
        context = examples['context'][i] if examples['context'][i] is not None else ""
        question = examples['question'][i]

        # 지문과 질문을 줄바꿈으로 연경
        if context:
            passage = f"{context}\n\n{question}"
        else:
            passage = question

        # 5개의 개별 보기 커럼을 하나의 리스트로 묶기
        choices = [
            examples['option#1'][i],
            examples['option#2'][i],
            examples['option#3'][i],
            examples['option#4'][i],
            examples['option#5'][i],
        ]

        # 정답 라벨 가져오기(만약 gold가 1~5 숫자라면 인덱스로 맞추기 위해 -1 처리가 필요할 수 있음
        label = int(examples['gold'][i]) - 1

        # 1단계 추론 때처럼 5개 보기를 각각 지문과 짝지어 토크나이징 처리
        encoded_inputs = tokenizer(
                text=[passage] * 5, # 대괄호를 써서 [지문, 지문, 지문, 지문, 지문] 리스트 생성
                text_pair=choices, # 5개의 보기 리스트, [보기1, 보기2, 보기3, 보기4, 보기5] 리스트
                max_length=512,
                padding=False, # 동적 패딩을 위해 여기서는 패딩을 하지 않음
                truncation=True,
        )

        # 각 보기별 input_ids, attention_mask 추출, [5,128] 2차원 텐서 형태
        # input_ids = [item['input_ids'] for item in encoded_inputs]
        # attention_mask = [item['attention_mask'] for item in encoded_inputs]

        # encoded_inputs는 이제 dict 형태이며, 각 키에 length 5짜리 리스트가 담겨 나온다
        batch_input_ids.append(encoded_inputs['input_ids'])
        batch_attention_mask.append(encoded_inputs['attention_mask'])
        batch_labels.append(label)
    return {
        "input_ids": batch_input_ids,
        "attention_mask": batch_attention_mask,
        "labels": batch_labels,
    }

# 학습 및 평가 데이터 샘플링(속도 최적화)
# train_sample_size = 300 # 전체 748
# eval_sample_size = 80 # 전체 188

# train_subset = train_dataset.select(range(min(train_sample_size, len(train_dataset))))
# eval_subset = eval_dataset.select(range(min(eval_sample_size, len(eval_dataset))))

# train/eval 데이터셋에 map 일괄 적용
encoded_train_dataset = train_dataset.map(
    function=preprocess_function,
    batched=True,
    remove_columns=train_dataset.column_names,
)
encoded_eval_dataset = eval_dataset.map(
    function=preprocess_function,
    batched=True,
    remove_columns=eval_dataset.column_names,
)

# 첫번째 데이터 샘플의 구조 확인
sample = encoded_train_dataset[0]
print(f"input_ids 길이(보기 개수): {len(sample['input_ids'])}")
print(f"첫 번째 보기의 토큰 길이: {len(sample['input_ids'][0])}")
print(f"labels 값: {sample['labels']}")

print(f"최종 전처리 완료된 학습 데이터 수: {len(encoded_train_dataset)}")
print(f"최종 전처리 완료된 평가 데이터 수: {len(encoded_eval_dataset)}")

Map:   0%|          | 0/748 [00:00<?, ? examples/s]

Map:   0%|          | 0/188 [00:00<?, ? examples/s]

input_ids 길이(보기 개수): 5
첫 번째 보기의 토큰 길이: 256
labels 값: 1
최종 전처리 완료된 학습 데이터 수: 748
최종 전처리 완료된 평가 데이터 수: 188


In [13]:
# 6. 다중선택(MultipleChoice) 모데을 위한 동적 패딩 전용 커스텀 데이터 콜레이터(Data Collator) 정의
# 배치내 데이터들을 1차원으로 펼쳐서(flatten) 가장 긴 길이에 맞게 동적 패딩을 수행한 뒤,
# 모델이 요구하는 [batch_size, num_choice, sequence] 3차원 텐서로 재조립 한다
import torch
from dataclasses import dataclass
from transformers.tokenization_utils_base import PreTrainedTokenizerBase, PaddingStrategy
from typing import Optional, Union

# 1. 다중선택 전용 데이터 콜레이터 (안정적인 딕셔너리 병합 방식)
@dataclass
class DataCollatorForMultipleChoice:
    tokenizer: PreTrainedTokenizerBase
    padding: Union[bool, str, PaddingStrategy] = True
    max_length: Optional[int] = None
    pad_to_multiple_of: Optional[int] = None

    def __call__(self, features):
        label_name = "label" if "label" in features[0] else "labels"
        labels = [feature.pop(label_name) for feature in features]
        batch_size = len(features)
        num_choices = len(features[0]["input_ids"])
        
        # 3차원 구조를 1차원 리스트로 평탄화
        flattened_features = [
            {k: [f[k][i] for f in features] for k in features[0].keys()}
            for i in range(num_choices)
        ]
        
        # 보기별로 합치기
        flattened_features = [
            {k: [item for sublist in flattened_features[i][k] for item in sublist] if isinstance(flattened_features[i][k][0], list) else flattened_features[i][k] for k in flattened_features[i].keys()}
            for i in range(num_choices)
        ]
        
        # 더 간단하게 표준 허깅페이스 공식 콜레이터 방식으로 평탄화 수행:
        # (아래 표준 구조가 가장 트러블이 없습니다)
        
        return self.collate_batch(features, labels, batch_size, num_choices)

    def collate_batch(self, features, labels, batch_size, num_choices):
        # 1차원 리스트로 각각 풀헤치기
        flattened_features = []
        for feature in features:
            for i in range(num_choices):
                flattened_features.append({
                    k: v[i] for k, v in feature.items()
                })
        
        # 동적 패딩 적용
        batch = self.tokenizer.pad(
            flattened_features,
            padding=self.padding,
            max_length=self.max_length,
            pad_to_multiple_of=self.pad_to_multiple_of,
            return_tensors="pt",
        )
        
        # 3차원 텐서로 재조립 [Batch, Num_Choices, Seq_Len]
        batch = {k: v.view(batch_size, num_choices, -1) for k, v in batch.items()}
        batch["labels"] = torch.tensor(labels, dtype=torch.long)
        return batch

# 2. 콜레이터 인스턴스 재생성
data_collator = DataCollatorForMultipleChoice(tokenizer=tokenizer)

In [14]:
# 7. 트레이너(Trainer) 설정
from transformers import AutoModelForMultipleChoice, TrainingArguments, Trainer

# 1) 모델 이름 지정 및 사전학습 모델 로드
model_name = "klue/bert-base"
model = AutoModelForMultipleChoice.from_pretrained(model_name)
model.to(device)

# 2) 학습 파라미터 설정
training_args = TrainingArguments(
    # RTX 3060 VRAM 안정권 (~4GB 유지)
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    learning_rate=2e-5,
    num_train_epochs=5,
    eval_strategy="epoch",
    logging_strategy="steps",
    logging_steps=30,
    # 최신 버전의 transformers 라이브러리에서는 TrainingArguments에서 logging_dir 인자가 제거되었거나 내부적으로 다르게 처리되도록 
    # 변경되었습니다. (TensorBoard 로그 디렉토리는 기본적으로 output_dir 하위에 자동으로 생성됩니다.)
    # logging_dir="./content/logs",
    save_strategy="epoch",
    output_dir="./content/csatqa_bert_ckpt",
    report_to="tensorboard",
    weight_decay=0.01,
    load_best_model_at_end=True, # 검증 손실이 가장 낮았던 최고의 체크포인트 자동 저장
    # 일반적인 딥러닝 연산은 32비트 부동소수점(FP32) 방식을 사용, 하지만 정밀도가 높은 만큼 메모리를 많이 먹고 연산 속도도 상대적으로 느림
    # fp16=True를 켜면 중요한 가중치 업데이트 등 꼭 필요한 순간만 32비트 고정밀도를 유지하고, 대다수의 무거운 텐서 연산은 16비트 반정밀도(FP16)로 처리
    fp16=True, # gpu RTX 3060 혼합 정밀도 가속
)

# 3) Trainer 선어
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=encoded_train_dataset,
    eval_dataset=encoded_eval_dataset,
    # 최신 버전의 transformers 라이브러리(5.x)에서는 Trainer 초기화 시 tokenizer 인자가 공식적으로 제거되었습니다. 
    # (토크나이저는 이제 TrainingArguments나 DataCollator 단계에서 처리되므로 Trainer에 직접 넘겨주지 않아도 됩니다.)
    # tokenizer=tokenizer,
    data_collator=data_collator, # 커스텀 콜레이터 장착
)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForMultipleChoice LOAD REPORT from: klue/bert-base
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [15]:
# 8. 학습 실행
# 파인튜닝 학습 시장, 수능 국어 5지선다 파인튜닝 학습 진행
print("RTX 3060 GPU, 수능 QA 모델 파인튜닝을 시작합니다...")
trainer.train()
print("파인튜닝 완료!")

RTX 3060 GPU, 수능 QA 모델 파인튜닝을 시작합니다...


Epoch,Training Loss,Validation Loss
1,1.606641,1.624065
2,1.462240,1.655646
3,0.977878,1.817938
4,0.638984,2.129919
5,0.375268,2.276036


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

파인튜닝 완료!


In [16]:
# 9. 모델 저장 
save_path = "./content/drive/MyDrive/trained_csatqa_bert_model"
trainer.save_model(save_path)
tokenizer.save_pretrained(save_path)

print(f"모델이 안전하게 저장되었습니다: {save_path}")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

모델이 안전하게 저장되었습니다: ./content/drive/MyDrive/trained_csatqa_bert_model
